# Stage 1: Pretraining on WikiText-2

Pretrain the LSTM as a general English language model on WikiText-2.

The text is one long token stream cut into batches of BPTT-length chunks, and the model predicts the next word at every position (truncated backpropagation through time).

**Output:** `artifacts/pretrained_lstm.pt`, used by `train.ipynb` for fine-tuning.

Run this notebook from the project root. It takes about 35 minutes on CPU. The pretrained model is already in `artifacts/`, so this step is optional.

In [ ]:
import json
import math
import re
import time
import urllib.request
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn

from nextword.model import LSTM, PAD, UNK, UNK_IDX, split_sentences, text_to_indices

ROOT = Path.cwd()
CHECKPOINT = ROOT / 'artifacts' / 'pretrained_lstm.pt'
REPORTS = ROOT / 'reports'

SEED = 42
VOCAB_SIZE = 12000
MODEL_CONFIG = dict(embed_dim=256, hidden_dim=256, num_layers=1, dropout=0.3, tie_weights=True)
BATCH_SIZE = 64
BPTT = 35
LR = 0.002
EPOCHS = 4

## 1. Data: WikiText-2 (raw)

The parquet files come from https://huggingface.co/datasets/Salesforce/wikitext and are downloaded on first use into `data/wikitext/` (not committed to git).

In [ ]:
URL = 'https://huggingface.co/datasets/Salesforce/wikitext/resolve/main/wikitext-2-raw-v1/{split}-00000-of-00001.parquet'
DATA_DIR = ROOT / 'data' / 'wikitext'


def detokenize(text):
    # WikiText splits numbers/hyphens as "1 @,@ 000" and "role @-@ playing"
    text = re.sub(r' @([-,.])@ ', r'\1', text)
    return re.sub(r' ([.,;:!?\'")])', r'\1', text)


def load_sentences(split):
    """Tokenized sentences for 'train' or 'validation', cached after the first run."""
    cache = DATA_DIR / f'wikitext2-{split}.tokens.pt'
    if cache.exists():
        return torch.load(cache)
    parquet = DATA_DIR / f'wikitext2-{split}.parquet'
    if not parquet.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print(f'Downloading WikiText-2 {split} ...')
        urllib.request.urlretrieve(URL.format(split=split), parquet)
    paragraphs = [t for t in pd.read_parquet(parquet)['text'] if t.strip() and not t.strip().startswith('=')]
    sentences = split_sentences('\n'.join(detokenize(p.strip()) for p in paragraphs))
    torch.save(sentences, cache)
    return sentences


def build_general_vocab(sentences, max_size):
    counts = Counter(t for s in sentences for t in s)
    vocab = {PAD: 0, UNK: 1}
    for word, _ in counts.most_common(max_size - len(vocab)):
        vocab[word] = len(vocab)
    return vocab

In [ ]:
torch.manual_seed(SEED)
train_s, val_s = load_sentences('train'), load_sentences('validation')
vocab = build_general_vocab(train_s, VOCAB_SIZE)
print(f'{len(train_s):,} train sentences, {len(val_s):,} validation sentences')
print('Example:', ' '.join(train_s[0]))

## 2. Batching for truncated BPTT

All sentences are joined into one token stream and split into `BATCH_SIZE` parallel rows. Training walks along the rows in chunks of `BPTT` steps, carrying the LSTM state across chunks but cutting the gradient.

In [ ]:
def to_stream(sentences, vocab):
    ids = [i for s in sentences for i in text_to_indices(s, vocab)]
    return torch.tensor(ids, dtype=torch.long)


def batchify(stream, batch_size):
    n = stream.size(0) // batch_size
    return stream[:n * batch_size].view(batch_size, n)          # (batch, time)


def run_epoch(model, data, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    state, total, count = None, 0.0, 0
    with torch.set_grad_enabled(training):
        for i in range(0, data.size(1) - 1, BPTT):
            seq = min(BPTT, data.size(1) - 1 - i)
            x, y = data[:, i:i + seq], data[:, i + 1:i + 1 + seq]
            if state is not None:
                state = tuple(s.detach() for s in state)       # truncated BPTT
            logits, state = model.forward_all(x, state)
            loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                optimizer.step()
            n = (y != UNK_IDX).sum().item()
            total += loss.item() * n
            count += n
    return total / count

In [ ]:
train, val = batchify(to_stream(train_s, vocab), BATCH_SIZE), batchify(to_stream(val_s, vocab), BATCH_SIZE)
print(f'WikiText-2: {train.numel():,} train tokens, {val.numel():,} val tokens, vocab {len(vocab):,}')

## 3. Model

Embedding(256) → LSTM(256) → Linear, with tied input/output embeddings (Press & Wolf, 2017) and dropout 0.3. Adam with a learning rate that is multiplied by 0.6 after every epoch.

In [ ]:
model = LSTM(len(vocab), **MODEL_CONFIG)
criterion = nn.CrossEntropyLoss(ignore_index=UNK_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.6)
print(model)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')

## 4. Training

In [ ]:
history = []
for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_loss = run_epoch(model, train, criterion, optimizer)
    val_loss = run_epoch(model, val, criterion)
    scheduler.step()
    history.append({'epoch': epoch, 'train_ppl': round(math.exp(train_loss), 2),
                    'val_ppl': round(math.exp(val_loss), 2)})
    print(f'Epoch {epoch}  train ppl {math.exp(train_loss):7.1f}  val ppl {math.exp(val_loss):7.1f}'
          f'  ({time.time() - start:.0f}s)', flush=True)

In [ ]:
hist = pd.DataFrame(history).set_index('epoch')
ax = hist.plot(marker='o', figsize=(6, 3.2), title='WikiText-2 perplexity (lower is better)')
ax.set_ylabel('perplexity')
ax.grid(alpha=0.25)
plt.show()

## 5. Save the pretrained model

In [ ]:
CHECKPOINT.parent.mkdir(exist_ok=True)
torch.save({'state_dict': model.state_dict(), 'vocab': vocab, 'model_config': MODEL_CONFIG}, CHECKPOINT)
REPORTS.mkdir(exist_ok=True)
(REPORTS / 'pretrain_metrics.json').write_text(json.dumps(
    {'dataset': 'WikiText-2 (raw)', 'train_tokens': train.numel(), 'val_tokens': val.numel(),
     'vocab_size': len(vocab), 'model': MODEL_CONFIG, 'batch_size': BATCH_SIZE, 'bptt': BPTT,
     'lr': LR, 'history': history}, indent=2))
print(f'Saved pretrained model to {CHECKPOINT}')